# Wind turbine outage: wind speed prediction pipeline

1. Load data -> `ds_train`, `ds_test`, `ds_result`
2. Power curves
3. Fit candidate models on `ds_train`
4. Evaluate every model on `ds_test`
5. Pick the best model
6. Retrain it on train + test, predict `ds_result`
7. Energy loss and submission export

In [12]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from get_dataset import get_data, target_turbines, reference_turbines
from power_curves import POWER_CURVE, energy_kwh, powercurve
from models import neighbour_mean, neighbour_mean_params
from evaluation import evaluate, select_best

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load data

In [13]:
ds_train, ds_test, ds_result = get_data(
    data_dir="data",
    test_start="2026-09-04",
    test_end=None,            # None = to the end of the data; "2026-09-09" = README shown test period
    result_period=None,       # None = auto-detect the outage; or ("2026-09-01 00:00", "2026-09-03 23:50")
    use_non_complete=False,   # False = train only on rows where every turbine has a wind speed
)

for name, ds in [("train", ds_train), ("test", ds_test), ("result", ds_result)]:
    print(f"{name:7s} {ds.index[0]} -> {ds.index[-1]}  {len(ds):5d} rows")
print("targets:   ", target_turbines(ds_train))
print("references:", reference_turbines(ds_train))

train   2026-07-01 00:00:00 -> 2026-08-31 23:40:00   7622 rows
test    2026-09-04 00:00:00 -> 2026-09-30 23:50:00   3888 rows
result  2026-09-01 00:00:00 -> 2026-09-03 23:50:00    432 rows
targets:    ['T1', 'T2', 'T3']
references: ['H1', 'H2', 'H3', 'H4', 'H5', 'O1', 'O2']


In [14]:
ds_train.head()

turbine                     H1                                     \
variable            wind_speed wind_direction ambient_temperature   
timestamp                                                           
2026-07-01 00:00:00   5.631282       51.43747            16.91814   
2026-07-01 00:20:00   4.966313       99.29465            17.21615   
2026-07-01 00:30:00   4.984415       89.38460            17.40417   
2026-07-01 00:40:00   5.017492       83.53693            17.29220   
2026-07-01 00:50:00   5.199528       82.57381            17.39974   

turbine                                                    H2                 \
variable            air_pressure wind_speed_stddev wind_speed wind_direction   
timestamp                                                                      
2026-07-01 00:00:00     993.5333          0.846214   6.253799       50.91308   
2026-07-01 00:20:00     993.6462          0.799924   5.158716       91.72142   
2026-07-01 00:30:00     993.3114          0.639666   5.016369       83.72660   
2026-07-01 00:40:00     993.2694          0.971072   5.426286       80.75471   
2026-07-01 00:50:00     993.1996          1.024312   5.067235       77.03947   

turbine                                                                 ...  \
variable            ambient_temperature air_pressure wind_speed_stddev  ...   
timestamp                                                               ...   
2026-07-01 00:00:00            16.61824     991.0911          0.712409  ...   
2026-07-01 00:20:00            16.90862     991.1936          1.077165  ...   
2026-07-01 00:30:00            17.14555     990.9982          0.802761  ...   
2026-07-01 00:40:00            17.06121     990.8621          0.868725  ...   
2026-07-01 00:50:00            16.99631     990.9300          1.199142  ...   

turbine                         T2                                   \
variable            wind_direction ambient_temperature air_pressure   
timestamp                                                             
2026-07-01 00:00:00       110.3659            17.41865     1001.844   
2026-07-01 00:20:00       135.3214            17.73365     1001.799   
2026-07-01 00:30:00       215.6006            18.06832     1001.923   
2026-07-01 00:40:00       248.4127            18.14267     1001.960   
2026-07-01 00:50:00       280.7722            18.41759     1001.839   

turbine                                       T3                 \
variable            wind_speed_stddev wind_speed wind_direction   
timestamp                                                         
2026-07-01 00:00:00          0.681208   4.829981       79.34425   
2026-07-01 00:20:00          0.923993   2.448443      123.71370   
2026-07-01 00:30:00          0.273377   1.628271      198.56650   
2026-07-01 00:40:00          0.618901   0.920418      196.12830   
2026-07-01 00:50:00          0.782594   0.887650      231.42650   

turbine                                                                 \
variable            ambient_temperature air_pressure wind_speed_stddev   
timestamp                                                                
2026-07-01 00:00:00            17.46686     1003.363          0.807648   
2026-07-01 00:20:00            17.96162     1003.504          0.698420   
2026-07-01 00:30:00            17.93876     1003.641          0.480172   
2026-07-01 00:40:00            17.88211     1003.583          0.404659   
2026-07-01 00:50:00            18.17443     1003.435          0.394133   

turbine             complete  
variable                      
timestamp                     
2026-07-01 00:00:00     True  
2026-07-01 00:20:00     True  
2026-07-01 00:30:00     True  
2026-07-01 00:40:00     True  
2026-07-01 00:50:00     True  

[5 rows x 51 columns]

In [15]:
# share of missing wind speed per turbine and set
pd.DataFrame({
    name: ds.xs("wind_speed", axis=1, level=1).isna().mean()
    for name, ds in [("train", ds_train), ("test", ds_test), ("result", ds_result)]
}).round(3)

,train,test,result
turbine,,,
H1,0.0,0.012,0.0
H2,0.0,0.006,0.0
H3,0.0,0.009,0.0
H4,0.0,0.010,0.0
H5,0.0,0.008,0.0
O1,0.0,0.008,0.0
O2,0.0,0.005,0.0
T1,0.0,0.009,1.0
T2,0.0,0.005,1.0


## 2. Power curves

In [16]:
# one curve for T1, T2 and T3 (identical files): wind speed (m/s) in, power (kW) out,
# on the straight line between the two neighbouring points of the 0.5 m/s table
powercurve(7.25)

np.float64(1887.0)

In [17]:
# 0 below cut-in, 4500 kW rated, ramp-down above 20 m/s, 0 above 26 m/s (cut-out)
wind = [2.5, 3.0, 3.25, 7.25, 12.0, 21.0, 26.0, 26.5]
pd.DataFrame({"power_kw": powercurve(wind)}, index=pd.Index(wind, name="wind_speed"))

,power_kw
wind_speed,
2.50,0.0
3.00,31.0
3.25,76.5
7.25,1887.0
12.00,4500.0
21.00,4307.0
26.00,2529.0
26.50,0.0


## 3. Models

**neighbour_mean** - same method as `4_Wind park T - all turbines calculation.xlsx`:
1. Average the wind speed of the 7 neighbouring turbines (H1-H5, O1, O2) at every 10-minute step.
2. Fit one straight line per target turbine on the training data: `T = intercept + slope * average`.
3. Predict with that line, never below 0 m/s.

In [18]:
from models import (
    neighbour_mean,
    neighbour_mean_params,
    best_reference_linear,
    best_reference_params,
    top3_multiple_linear,
    ridge_all_references,
)

models = {
    "neighbour_mean": neighbour_mean,
    "best_reference_linear": best_reference_linear,
    "top3_multiple_linear": top3_multiple_linear,
    "ridge_all_references": ridge_all_references,
}

In [19]:
best_reference_params(ds_train)

,reference,correlation,slope,intercept,r2,n
T1,O1,0.710833,0.599948,1.649006,0.505284,7622
T2,O2,0.775913,0.655118,1.706032,0.602041,7622
T3,O1,0.747652,0.604736,1.572053,0.558984,7622


## 4. Evaluate on the test set

In [ ]:
# target turbines are hidden before predicting, so the test period looks exactly like an outage
scores = pd.concat({name: evaluate(m, ds_train, ds_test) for name, m in models.items()}, names=["model", "turbine"])
scores.round(3)

wind_rmse  wind_mae  power_rmse  power_mae  \
model                 turbine                                               
neighbour_mean        T1           1.380     1.065     820.693    551.538   
                      T2           1.117     0.873     693.197    465.094   
                      T3           1.183     0.920     738.470    486.403   
best_reference_linear T1           1.536     1.194     902.701    606.926   
                      T2           1.322     1.043     794.122    538.143   
                      T3           1.361     1.062     818.543    543.662   
top3_multiple_linear  T1           1.383     1.064     824.959    549.577   
                      T2           1.127     0.874     700.233    463.999   
                      T3           1.188     0.921     732.503    484.726   
ridge_all_references  T1           1.378     1.057     820.448    545.508   
                      T2           1.097     0.854     683.547    454.046   
                      T3           1.168     0.908     727.233    479.426   

                               energy_dev_pct  coverage  
model                 turbine                            
neighbour_mean        T1               -8.626     1.000  
                      T2               -8.595     1.000  
                      T3              -12.835     1.000  
best_reference_linear T1               -9.385     0.992  
                      T2              -10.199     0.995  
                      T3              -14.241     0.992  
top3_multiple_linear  T1               -6.949     1.000  
                      T2               -6.836     1.000  
                      T3               -9.591     1.000  
ridge_all_references  T1               -6.333     1.000  
                      T2               -6.033     1.000  
                      T3              -10.417     1.000

## 5. Pick the best model

In [9]:
best_name = select_best(scores)   # lowest mean wind RMSE over T1-T3; select_best(scores, "energy_dev_pct") for energy
best_name

'neighbour_mean'

## 6. Retrain on train + test, predict the outage

In [10]:
# retrain the best model on more data, including the days right after the outage
ds_full = pd.concat([ds_train, ds_test]).sort_index()
ds_full = ds_full[ds_full["complete"]]   # same row filter as use_non_complete=False
results_sub = models[best_name](ds_full, ds_result)

print(f"outage {ds_result.index[0]} -> {ds_result.index[-1]}, missing predictions: {results_sub.isna().sum().sum()}")
results_sub.describe().round(2)

outage 2026-09-01 00:00:00 -> 2026-09-03 23:50:00, missing predictions: 0


,T1,T2,T3
count,432.00,432.00,432.00
mean,6.07,6.52,6.10
std,1.21,1.40,1.25
min,3.20,3.19,3.12
25%,5.11,5.41,5.11
50%,6.22,6.70,6.25
75%,6.92,7.51,6.98
max,8.70,9.58,8.82


## 7. Energy loss and submission

In [11]:
PRICE_EUR_PER_MWH = 80   # same price as the Summary sheet of the Excel calculation

loss = pd.DataFrame({
    "mean_wind_ms": results_sub.mean(),
    "lost_kwh": pd.Series({t: energy_kwh(results_sub[t]) for t in results_sub}),
})
loss["lost_mwh"] = loss["lost_kwh"] / 1000
loss["lost_eur"] = loss["lost_mwh"] * PRICE_EUR_PER_MWH
loss.loc["Total", ["lost_kwh", "lost_mwh", "lost_eur"]] = loss[["lost_kwh", "lost_mwh", "lost_eur"]].sum()
loss.round(1)

,mean_wind_ms,lost_kwh,lost_mwh,lost_eur
T1,6.1,87617.0,87.6,7009.4
T2,6.5,110783.6,110.8,8862.7
T3,6.1,89432.7,89.4,7154.6
Total,NaN,287833.3,287.8,23026.7


In [12]:
# TODO: confirm the submission format with the organisers
# results_sub.to_csv("submission.csv")